# Forward-Deployed AI Lab: Evaluation Walkthrough

This notebook demonstrates the credential-free path: a grounded read, an approval-gated Salesforce-style action, the golden-set release gate, and the adversarial test suite. All data is synthetic.


In [ ]:
from pathlib import Path
from pprint import pprint

from forward_deployed_ai_lab.app import build_container
from forward_deployed_ai_lab.evaluation import run_benchmark, run_red_team
from forward_deployed_ai_lab.models.domain import ActionKind, AssistRequest

container = build_container()
print(container.settings.app_name)


## 1. Grounded, read-only enterprise request


In [ ]:
safe_response = container.orchestrator.run(
    AssistRequest(query='What is the response target for a Priority 1 incident?')
)
pprint(safe_response.model_dump(mode='json'))


## 2. Consequential action routed to a named human

The orchestrator creates a proposal and approval reference; it does not execute the system-of-record write.


In [ ]:
write_response = container.orchestrator.run(
    AssistRequest(
        query='Close the case after documenting the approved resolution.',
        case_id='500000000000001',
        requested_action=ActionKind.CLOSE_CASE,
    )
)
print('decision:', write_response.policy.decision)
print('approval:', write_response.approval_id)
pprint(write_response.proposed_action.model_dump(mode='json'))


## 3. Reproducible golden-set release gate


In [ ]:
benchmark = run_benchmark(
    container.orchestrator,
    golden_set_path=Path(container.settings.data_dir) / 'eval/golden_set.json',
    output_path=Path(container.settings.artifact_dir) / 'evaluation-report.json',
)
pprint(benchmark['metrics'])
pprint(benchmark['release_gate'])


## 4. Adversarial controls


In [ ]:
red_team = run_red_team(
    container.orchestrator,
    prompts_path=Path(container.settings.data_dir) / 'red_team/prompts.json',
    output_path=Path(container.settings.artifact_dir) / 'red-team-report.json',
)
pprint(red_team['metrics'])


## Production extensions

The repository keeps RAGAS, DeepEval, MLflow, vector databases, LangGraph persistence, Spark/Databricks, and cloud AI platforms optional. This makes the default notebook reproducible while preserving clear interfaces for production evaluation, orchestration, data preparation, and deployment.
